# Import Library

In [23]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

os.makedirs("output", exist_ok=True)
os.makedirs("data", exist_ok=True)

# Load Data

In [35]:
import pandas as pd

# Jalur file relatif sesuai struktur folder di Colab Anda
path_data = "/content/data/case3_ecommerce_transactions - case3_ecommerce_transactions.csv.csv"
df_mentah = pd.read_csv(path_data)

print("--- Dimension Data ---")
print(f"Jumlah Baris: {df_mentah.shape[0]}, Jumlah Kolom: {df_mentah.shape[1]}")

print("\n--- Info Struktur Kolom ---")
df_mentah.info()

print("\n--- Statistik Deskriptif Numerik ---")
df_mentah.describe()

--- Dimension Data ---
Jumlah Baris: 10000, Jumlah Kolom: 14

--- Info Struktur Kolom ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   transaction_id     10000 non-null  object 
 1   user_id            9911 non-null   object 
 2   product_id         10000 non-null  object 
 3   kategori_produk    10000 non-null  object 
 4   nama_produk        9804 non-null   object 
 5   harga_idr          9702 non-null   float64
 6   kuantitas          9805 non-null   float64
 7   diskon_pct         10000 non-null  int64  
 8   rating_pembeli     9463 non-null   float64
 9   sentimen_ulasan    9601 non-null   object 
 10  tanggal_pembelian  9813 non-null   object 
 11  metode_pembayaran  10000 non-null  object 
 12  kota_pembeli       9724 non-null   object 
 13  status_retur       10000 non-null  object 
dtypes: float64(3), int64(1), obje

,harga_idr,kuantitas,diskon_pct,rating_pembeli
count,9.702000e+03,9805.000000,10000.000000,9463.000000
mean,2.513745e+06,3.011117,13.204500,2.997633
std,1.434086e+06,1.418814,10.544657,1.150633
min,1.002500e+04,1.000000,0.000000,1.000000
25%,1.262930e+06,2.000000,5.000000,2.000000
50%,2.499292e+06,3.000000,15.000000,3.000000
75%,3.755658e+06,4.000000,20.000000,4.000000
max,4.999941e+06,5.000000,30.000000,5.000000


In [25]:
df_mentah.isnull().sum()

,0
transaction_id,0
user_id,89
product_id,0
kategori_produk,0
nama_produk,196
harga_idr,298
kuantitas,195
diskon_pct,0
rating_pembeli,537
sentimen_ulasan,399


# Fungsi Audit Kualitas Data (cek_kualitas)

In [26]:
def cek_kualitas(d, kolom_num, kolom_grup, kunci):
    g  = d.groupby(kolom_grup, observed=True)[kolom_num]
    q1 = g.transform(lambda s: s.quantile(0.25))
    q3 = g.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    outlier = (d[kolom_num] < q1 - 1.5 * iqr) | (d[kolom_num] > q3 + 1.5 * iqr)

    return pd.Series({
        'n_baris':    len(d),
        'n_duplikat': d.duplicated(subset=kunci).sum(),
        'n_missing':  d.isnull().sum().sum(),
        'n_outlier':  outlier.sum(),
    })

# Run audit awal pada data mentah
kolom_num_audit = 'harga_idr'
kolom_grup_audit = 'kategori_produk'
kunci_audit = 'transaction_id'

res_sebelum = cek_kualitas(df_mentah, kolom_num_audit, kolom_grup_audit, kunci_audit)
print("Hasil Audit Sebelum Pembersihan:")
print(res_sebelum)

Hasil Audit Sebelum Pembersihan:
n_baris       10000
n_duplikat        0
n_missing      2177
n_outlier         0
dtype: int64


# Pembersihan Data Berurutan

In [27]:
df_bersih = df_mentah.copy()

# 1. Hapus Duplikat berdasarkan ID Transaksi
df_bersih = df_bersih.drop_duplicates(subset=[kunci_audit])

# 2. Penanganan Missing Value
# Hapus baris tanpa user_id karena tidak dapat dianalisis untuk RFM
df_bersih = df_bersih.dropna(subset=['user_id'])

# Imputasi kolom numerik dengan nilai median per kategori produk
for col in ['harga_idr', 'kuantitas', 'rating_pembeli']:
    df_bersih[col] = df_bersih.groupby('kategori_produk')[col].transform(lambda s: s.fillna(s.median()))

# Imputasi kolom kategorikal & teks
df_bersih['nama_produk'] = df_bersih['nama_produk'].fillna('Unknown Product')
df_bersih['sentimen_ulasan'] = df_bersih['sentimen_ulasan'].fillna('Netral')
df_bersih['kota_pembeli'] = df_bersih['kota_pembeli'].fillna('Lainnya')
df_bersih['tanggal_pembelian'] = df_bersih['tanggal_pembelian'].ffill().bfill()

# Feature Engineering: Total Nilai Transaksi
df_bersih['total_pembayaran'] = df_bersih['harga_idr'] * df_bersih['kuantitas'] * (1 - df_bersih['diskon_pct'] / 100)

# 3. Penanganan Outlier (Clipping IQR per Kategori Produk)
g = df_bersih.groupby(kolom_grup_audit, observed=True)[kolom_num_audit]
q1 = g.transform(lambda s: s.quantile(0.25))
q3 = g.transform(lambda s: s.quantile(0.75))
iqr = q3 - q1
df_bersih[kolom_num_audit] = df_bersih[kolom_num_audit].clip(
    lower=q1 - 1.5 * iqr,
    upper=q3 + 1.5 * iqr
)

# 4. Penyesuaian Tipe Data
df_bersih['tanggal_pembelian'] = pd.to_datetime(df_bersih['tanggal_pembelian'])
df_bersih['kuantitas'] = df_bersih['kuantitas'].astype(int)

# Tabel Komparasi Audit Sebelum vs Sesudah
laporan = pd.DataFrame({
    'sebelum': res_sebelum,
    'sesudah': cek_kualitas(df_bersih, kolom_num_audit, kolom_grup_audit, kunci_audit),
})

print("\n--- Tabel Laporan Kualitas Data ---")
print(laporan)

# Menyimpan laporan kualitas
laporan.to_csv("laporan_kualitas.csv")


--- Tabel Laporan Kualitas Data ---
            sebelum  sesudah
n_baris       10000     9911
n_duplikat        0        0
n_missing      2177        0
n_outlier         0        0


In [28]:
df_bersih.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9911 entries, 0 to 9998
Data columns (total 15 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   transaction_id     9911 non-null   object        
 1   user_id            9911 non-null   object        
 2   product_id         9911 non-null   object        
 3   kategori_produk    9911 non-null   object        
 4   nama_produk        9911 non-null   object        
 5   harga_idr          9911 non-null   float64       
 6   kuantitas          9911 non-null   int64         
 7   diskon_pct         9911 non-null   int64         
 8   rating_pembeli     9911 non-null   float64       
 9   sentimen_ulasan    9911 non-null   object        
 10  tanggal_pembelian  9911 non-null   datetime64[ns]
 11  metode_pembayaran  9911 non-null   object        
 12  kota_pembeli       9911 non-null   object        
 13  status_retur       9911 non-null   object        
 14  total_pembaya

In [29]:
df_bersih.describe()

,harga_idr,kuantitas,diskon_pct,rating_pembeli,tanggal_pembelian,total_pembayaran
count,9.911000e+03,9911.000000,9911.000000,9911.000000,9911,9.911000e+03
mean,2.514278e+06,3.009888,13.203511,2.998386,2023-06-30 21:43:25.468671232,6.554766e+06
min,1.080400e+04,1.000000,0.000000,1.000000,2022-01-01 00:00:00,9.736300e+03
25%,1.305782e+06,2.000000,5.000000,2.100000,2022-09-29 12:00:00,2.440356e+06
50%,2.498760e+06,3.000000,15.000000,3.000000,2023-06-30 00:00:00,5.078298e+06
75%,3.711716e+06,4.000000,20.000000,3.900000,2024-03-31 00:00:00,9.748206e+06
max,4.999941e+06,5.000000,30.000000,5.000000,2024-12-31 00:00:00,2.474826e+07
std,1.410989e+06,1.405088,10.536030,1.120266,NaN,5.174456e+06


In [30]:
df_bersih.isnull().sum()

,0
transaction_id,0
user_id,0
product_id,0
kategori_produk,0
nama_produk,0
harga_idr,0
kuantitas,0
diskon_pct,0
rating_pembeli,0
sentimen_ulasan,0


# Analisis Data, Time Series, & RFM

In [31]:
# A. Groupby Lanjutan per Kategori Produk
groupby_kategori = df_bersih.groupby('kategori_produk').agg(
    total_transaksi=('transaction_id', 'count'),
    total_pendapatan=('total_pembayaran', 'sum'),
    rata_diskon=('diskon_pct', 'mean'),
    persentase_retur=('status_retur', lambda x: (x == 'Diretur').mean() * 100)
).sort_values(by='total_pendapatan', ascending=False)

print("--- Ringkasan per Kategori Produk ---")
print(groupby_kategori)

# B. Tren Waktu Bulanan
df_bersih['tahun_bulan'] = df_bersih['tanggal_pembelian'].dt.to_period('M')
tren_bulanan = df_bersih.groupby('tahun_bulan').agg(
    omset=('total_pembayaran', 'sum'),
    jumlah_transaksi=('transaction_id', 'count')
)

print("\n--- Tren Omset Bulanan ---")
print(tren_bulanan.tail(6))

# C. Analisis RFM (Recency, Frequency, Monetary)
max_date = df_bersih['tanggal_pembelian'].max()
rfm = df_bersih.groupby('user_id').agg(
    Recency=('tanggal_pembelian', lambda x: (max_date - x.max()).days),
    Frequency=('transaction_id', 'count'),
    Monetary=('total_pembayaran', 'sum')
).reset_index()

print("\n--- Ringkasan Statistik RFM ---")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe())

--- Ringkasan per Kategori Produk ---
                   total_transaksi  total_pendapatan  rata_diskon  \
kategori_produk                                                     
Fashion Pria                  1013      6.945968e+09    13.099704   
Perabot Rumah                 1032      6.824136e+09    13.173450   
Otomotif                      1027      6.684967e+09    13.252191   
Fashion Wanita                 964      6.559064e+09    13.309129   
Makanan & Minuman              964      6.536786e+09    13.345436   
Elektronik                     987      6.437458e+09    13.541033   
Mainan Anak                   1018      6.368863e+09    13.035363   
Buku                           974      6.323215e+09    13.044148   
Olahraga                       988      6.252709e+09    12.940283   
Kecantikan                     944      6.031119e+09    13.310381   

                   persentase_retur  
kategori_produk                      
Fashion Pria              22.112537  
Perabot Rumah      

# Persiapan Model tanpa Data Leakage

In [32]:
# Menentukan Fitur (X) dan Target (y)
target_col = 'status_retur'
feature_cols = [
    'kategori_produk', 'harga_idr', 'kuantitas', 'diskon_pct',
    'rating_pembeli', 'sentimen_ulasan', 'metode_pembayaran',
    'kota_pembeli', 'total_pembayaran'
]

X = df_bersih[feature_cols]
y = df_bersih[target_col]

# Split data 80:20 dengan Stratify pada target imbalanced
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Menentukan kolom numerik dan kategorikal
num_cols = ['harga_idr', 'kuantitas', 'diskon_pct', 'rating_pembeli', 'total_pembayaran']
cat_cols = ['kategori_produk', 'sentimen_ulasan', 'metode_pembayaran', 'kota_pembeli']

# Pipeline Transformasi
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# Fit HANYA pada X_train untuk mencegah leakage
X_train_prep = preprocessor.fit_transform(X_train)
X_test_prep = preprocessor.transform(X_test)

print(f"Ukuran X_train terproses: {X_train_prep.shape}")
print(f"Ukuran X_test terproses: {X_test_prep.shape}")

Ukuran X_train terproses: (7928, 41)
Ukuran X_test terproses: (1983, 41)


# Ekspor CSV Bersih & Kamus Data

In [33]:
# 1. Simpan Data Bersih (tanpa kolom temporer tahun_bulan)
if 'tahun_bulan' in df_bersih.columns:
    df_bersih = df_bersih.drop(columns=['tahun_bulan'])

df_bersih.to_csv("output/data_bersih.csv", index=False)

# 2. Ekspor Kamus Data
kamus_data = pd.DataFrame({
    'nama_kolom': df_bersih.columns,
    'tipe_data': df_bersih.dtypes.astype(str),
    'n_missing': df_bersih.isnull().sum().values,
    'n_unik': df_bersih.nunique().values
})

kamus_data.to_csv("output/kamus_data.csv", index=False)

print("--- Hasil Ekspor Berhasil ---")
print(kamus_data)

--- Hasil Ekspor Berhasil ---
                          nama_kolom       tipe_data  n_missing  n_unik
transaction_id        transaction_id          object          0    9911
user_id                      user_id          object          0    2893
product_id                product_id          object          0     500
kategori_produk      kategori_produk          object          0      10
nama_produk              nama_produk          object          0     100
harga_idr                  harga_idr         float64          0    9609
kuantitas                  kuantitas           int64          0       5
diskon_pct                diskon_pct           int64          0       7
rating_pembeli        rating_pembeli         float64          0      41
sentimen_ulasan      sentimen_ulasan          object          0      11
tanggal_pembelian  tanggal_pembelian  datetime64[ns]          0    1096
metode_pembayaran  metode_pembayaran          object          0       4
kota_pembeli            kota_pembe